## Download Necassary Libraries 

In [1]:
pip install torch transformers accelerate


This cell sets up the necessary components for text generation using the Hugging Face `transformers` library.

1.  **`AutoTokenizer.from_pretrained(model_id)`**: This loads the tokenizer associated with the specific model (`openai/gpt-oss-20b`). The tokenizer is responsible for converting raw text into numerical tokens that the model can understand.
2.  **`pipeline("text-generation", ...)`**: This creates a high-level pipeline object that abstracts away the complexity of the model inference process.
    *   `model=model_id`: Specifies which pre-trained model to load.
    *   `torch_dtype="auto"`: Automatically selects the optimal data type (like float16 or bfloat16) based on the available hardware to save memory and improve speed.
    *   `device_map="auto"`: Automatically distributes the model layers across available devices (CPU/GPU) to handle large models efficiently.

In [ ]:
from transformers import pipeline, AutoTokenizer

# Setup
model_id = "openai/gpt-oss-20b"
tokenizer = AutoTokenizer.from_pretrained(model_id)
pipe = pipeline("text-generation", model=model_id, torch_dtype="auto", device_map="auto")

This cell prepares and analyzes the input for the model.

1.  **`tokenizer.apply_chat_template(messages, ...)`**: This function formats the conversation history into a single string that matches the specific prompt structure required by the model.
    *   `messages`: A list of dictionaries representing the chat history (e.g., `{"role": "user", "content": "..."}`).
    *   `tokenize=False`: Returns the formatted string instead of converting it directly to token IDs.
    *   `add_generation_prompt=True`: Appends the specific tokens that signal the start of the assistant's turn, prompting the model to generate a response.

2.  **`tokenizer.encode(chat_input)`**: Converts the formatted string into a sequence of numerical token IDs that the model can process.

3.  **`tokenizer.decode([token_id])`**: Converts a numerical token ID back into its corresponding text string. This is used in the loop to visualize exactly how the input text is broken down into individual tokens.


In [ ]:
# Prepare input
messages = [{"role": "user", "content": "tell me a joke about cheese"}]
chat_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Tokenize input
input_ids = tokenizer.encode(chat_input)
print("INPUT TOKENS")
print("-" * 40)
print(f"{'Token':<20} | {'ID':<10}")
print("-" * 40)
for token_id in input_ids:
    token = tokenizer.decode([token_id])
    print(f"{repr(token):<20} | {token_id:<10}")


This cell executes the text generation and breaks down the model's output.

1.  **`pipe(messages, ...)`**: This runs the generation pipeline.
    *   `messages`: The input conversation history.
    *   `max_new_tokens=1024`: Limits the generated response to a maximum of 1024 new tokens.
    *   `return_full_text=True`: Ensures the output includes both the input prompt and the newly generated text.

2.  **`tokenizer.apply_chat_template(..., tokenize=False)`**: Converts the raw output structure back into a single formatted string, preserving special tokens.

3.  **`tokenizer.encode(assistant_part, add_special_tokens=False)`**: Converts the isolated assistant response back into token IDs for analysis, without adding extra start/end tokens.

4.  **Output Loop**: Iterates through the generated token IDs and decodes them one by one to visualize exactly how the model constructed the response token by token.


In [ ]:
# Generate
outputs = pipe(messages, max_new_tokens=1024, return_full_text=True)

# Get the full generated text with special tokens
full_generated = tokenizer.apply_chat_template(outputs[0]["generated_text"], tokenize=False)
# Extract just the new part (after the input)
assistant_part = full_generated[len(chat_input):]

# Tokenize output with special tokens
output_ids = tokenizer.encode(assistant_part, add_special_tokens=False)

print("\nOUTPUT TOKENS")
print("-" * 40)
print(f"{'Token':<20} | {'ID':<10}")
print("-" * 40)
for token_id in output_ids:
    token = tokenizer.decode([token_id])
    print(f"{repr(token):<20} | {token_id:<10}")